# Demo cấp 3 · ViHeritage-MMKG (tiếng Việt, nhóm tự xây)

Dùng đúng mã dùng chung X0 của notebook 10b, 10c, 10d (dữ liệu 08b, checkpoint 09 đã khóa, hai bộ giải thích). Chọn mô hình (GAT: che cạnh; AdaMF: học lại nhúng), cách diễn đạt câu (câu hỏi, câu mô tả, tiếng Anh) và một ca của báo cáo, hoặc tự nhập câu + thực thể neo + quan hệ. Đồ thị là các bộ ba train làm tăng điểm nhiều nhất (K* đọc từ `10b_run_info.json`).

**Cách dùng khi quay video:** Runtime → Run all. Ô cuối in ra một link `https://….gradio.live`: mở link ở tab mới (toàn màn hình).
Chọn ca trong danh sách là trang tự chạy; bấm **Bỏ giải thích S** trên đồ thị để xem ảnh rơi hạng. Link công khai và tự hết hạn,
tắt notebook (Runtime → Disconnect) sau khi quay. Đặt biến môi trường `XAI_DEMO_SHARE=0` nếu chỉ muốn xem trong notebook.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True)

## X0. Mã dùng chung của 10b, 10c, 10d (nguyên văn)

In [ ]:
# Ô X0 của notebook 10b, 10c, 10d (ViHeritage-MMKG), chép nguyên văn từ 10d_transfer_demo.ipynb trên Drive (bản 17/11/2026).
# Dùng để dựng notebook demo cấp 3 (demo/build_demo_notebooks.py). Không sửa tay: sửa ở notebook gốc rồi chép lại.
# X0. Mã dùng chung của 10b, 10c, 10d — giống hệt nhau ở ba notebook (SHARED_HASH được ghi vào kết quả và đối chiếu).
#     Nạp dữ liệu 08b, mã mô hình và checkpoint 09 đã khóa, chấm truy vấn như 09, và hai bộ giải thích:
#       AdaMF → E3 của cấp 2 (học lại nhúng cấu trúc của e* kiểu Kelpie), GAT → che cạnh trên đồ thị truyền tin (chính xác, không xấp xỉ).
import os, sys, re, json, glob, math, time, hashlib, subprocess, unicodedata, base64, io
from pathlib import Path
from datetime import datetime, timezone, timedelta
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
for _pkg, _mod in [("sentence-transformers", "sentence_transformers"), ("torch_geometric", "torch_geometric")]:
    try:
        __import__(_mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from IPython.display import display

def find_project_root():
    env = os.environ.get("MMKG_PROJECT_ROOT")
    if env:
        return Path(env)
    for p in ["/content/drive/MyDrive/MLGs - Project", "/content/drive/MyDrive/*/MLGs - Project",
              "/content/drive/Shareddrives/*/MLGs - Project", "/content/drive/MyDrive/.shortcut-targets-by-id/*/MLGs - Project"]:
        hits = sorted(glob.glob(p))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError("Không thấy thư mục 'MLGs - Project'.")

PROJECT_ROOT = find_project_root()
FINAL = PROJECT_ROOT / "0. final"
MMKG = FINAL / "data" / "MMKG"; PROC = MMKG / "processed"; REVIEW = MMKG / "review"
OUT09 = Path(os.environ.get("MMKG_OUT", FINAL / "results" / "MMKG"))
XOUT = OUT09 / "XAI_L3"; XOUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TZ = timezone(timedelta(hours=7))
torch.backends.cuda.matmul.allow_tf32 = False
SEEDS, MAIN_SEED = [2027, 2028, 2029], 2027
LANG, TEXT_VARIANT = "vi", "text_vi_1"         # câu dùng khi giải thích: cách nói tiếng Việt thứ nhất, α khóa của (mô hình, vi)
N_BOOT = 2000

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def write_json(obj, path):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=float), encoding="utf-8"); os.replace(tmp, path)

def stable_seed(*parts):
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:8], 16)

man = json.loads((PROC / "split_manifest.json").read_text(encoding="utf-8"))
for _f, _spec in man["artifacts"].items():
    assert sha(MMKG / _spec["path"]) == _spec["sha256"], f"File đã đổi sau 08b: {_f}"
HASHES = {f: s["sha256"] for f, s in man["artifacts"].items()}

# ---------- dữ liệu (đọc giống 09) ----------
def read_mapping(p):
    with open(p, encoding="utf-8") as f:
        n = int(f.readline()); rows = [x.rstrip("\n").rsplit("\t", 1) for x in f if x.strip()]
    assert len(rows) == n
    return [r[0] for r in rows]

def read_triples(p):
    with open(p) as f:
        n = int(f.readline()); a = np.loadtxt(f, dtype=np.int64, ndmin=2) if n else np.zeros((0, 3), np.int64)
    assert a.shape == (n, 3)
    return a                                                     # head tail relation

REPRO = PROC / "reproduction"
N, R = len(read_mapping(REPRO / "entity2id.txt")), len(read_mapping(REPRO / "relation2id.txt"))
SPL = {s: read_triples(REPRO / f"{s}2id.txt") for s in ["train", "valid", "test"]}
TRAIN = SPL["train"]
ents = pd.read_parquet(PROC / "entities.parquet"); rels = pd.read_csv(PROC / "relations.csv", keep_default_na=False)
LAB = dict(zip(ents.entity_index, ents.entity_label)); TYPE = dict(zip(ents.entity_index, ents.type))
PID = dict(zip(rels.relation_index, rels.pid)); RLAB = dict(zip(rels.relation_index, rels.label_vi))
DERIVED = set(rels.loc[rels.applies_to == "derived", "relation_index"].astype(int))
Q_ALL = pd.read_parquet(PROC / "queries.parquet"); REL_ALL = pd.read_parquet(PROC / "relevance.parquet")
FLT_ALL = pd.read_parquet(PROC / "query_filter.parquet"); pools = pd.read_parquet(PROC / "candidate_pools.parquet")
idx = pd.read_parquet(PROC / "image_feature_index.parquet"); IMGS = pd.read_parquet(PROC / "images.parquet").set_index("image_id")
img_feat = np.load(PROC / "clip_image_features.npy").astype("float32")
textual = torch.load(PROC / "textual.pt", map_location="cpu", weights_only=True).float()
ROW = idx.set_index("image_id").feature_row; ROW2IMG = idx.set_index("feature_row").image_id
IMG_ENT = idx.sort_values("feature_row").entity_index.to_numpy()
POOL = {s: np.sort(pools.loc[pools.split == s, "image_id"].map(ROW).to_numpy(np.int64)) for s in ["val", "test"]}
tr_rows = idx.loc[idx.retrieval_split == "train", "feature_row"].to_numpy()
train_visual = torch.zeros(N, 512); _cnt = torch.zeros(N)
train_visual.index_add_(0, torch.tensor(IMG_ENT[tr_rows]), torch.tensor(img_feat[tr_rows]))
_cnt.index_add_(0, torch.tensor(IMG_ENT[tr_rows]), torch.ones(len(tr_rows)))
train_visual = F.normalize(train_visual, dim=1)
availability = torch.stack([_cnt > 0, torch.ones(N, dtype=torch.bool)], dim=1)
IMG_N = img_feat / np.linalg.norm(img_feat, axis=1, keepdims=True)
TRAIN_KEYS = {tuple(t) for t in TRAIN.tolist()}
HELD_KEYS = {tuple(t) for t in np.concatenate([SPL["valid"], SPL["test"]]).tolist()}
FACTS_OF, PAIR = {}, {}
for _i, (_h, _t, _r) in enumerate(TRAIN.tolist()):
    FACTS_OF.setdefault(_h, []).append(_i); FACTS_OF.setdefault(_t, []).append(_i)
    PAIR.setdefault((min(_h, _t), max(_h, _t)), []).append(_i)
NEIGH = {e: {int(TRAIN[i, 0]) if int(TRAIN[i, 1]) == e else int(TRAIN[i, 1]) for i in fs} for e, fs in FACTS_OF.items()}
DEG = np.bincount(TRAIN[:, :2].ravel(), minlength=N)
REL_HEADS = {int(r): np.unique(TRAIN[TRAIN[:, 2] == r, 0]) for r in np.unique(TRAIN[:, 2])}
REL_TAILS = {int(r): np.unique(TRAIN[TRAIN[:, 2] == r, 1]) for r in np.unique(TRAIN[:, 2])}

def fact_text(fid=None, trip=None):
    h, t, r = (int(x) for x in (TRAIN[fid] if trip is None else trip))
    return f"{LAB[h]} –[{RLAB[r]}]→ {LAB[t]}"

def image_path(row):
    return MMKG / IMGS.loc[ROW2IMG[int(row)], "image_path"]

# ---------- câu truy vấn (như 09 T4) ----------
TXT = pd.read_csv(REVIEW / "queries_text.csv", encoding="utf-8-sig", keep_default_na=False, dtype=str)
TXT = TXT.set_index("text_group")
for _c in ["text_vi_1", "text_vi_2", "text_en"]:
    TXT[_c] = TXT[_c].map(lambda x: unicodedata.normalize("NFC", x).strip())

def qtext(q, variant=TEXT_VARIANT):
    return (getattr(q, variant) or q.text_vi_1).strip() if q.query_type == "direct" else TXT.loc[q.text_group, variant]

def split_queries(sp):
    q = Q_ALL[Q_ALL.split == sp].sort_values("query_id").reset_index(drop=True)
    rel = REL_ALL[REL_ALL.query_id.isin(q.query_id)].assign(row=lambda d: d.image_id.map(ROW))
    flt = FLT_ALL[FLT_ALL.query_id.isin(q.query_id)].assign(row=lambda d: d.image_id.map(ROW))
    pos = rel.groupby("query_id").row.apply(lambda x: np.unique(x.to_numpy(np.int64))).to_dict()
    drop = flt.groupby("query_id").row.apply(lambda x: set(x.to_numpy(np.int64).tolist())).to_dict()
    return q, pos, drop

# ---------- bộ mã hóa văn bản: đúng revision 09 đã ghim ----------
ENC = json.loads((OUT09 / "text_encoder.json").read_text())
_enc = None
def encode(texts):
    global _enc
    if _enc is None:
        from sentence_transformers import SentenceTransformer
        _enc = SentenceTransformer(ENC["name"], revision=ENC["revision"], device=str(DEVICE))
    return _enc.encode(list(texts), batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")

# ---------- mô hình: đúng mã nguồn và checkpoint 09 ----------
MODEL_SOURCE = (OUT09 / "model_source.py").read_text(encoding="utf-8")
GAT_SOURCE = (OUT09 / "gat_source.py").read_text(encoding="utf-8")
exec(MODEL_SOURCE); exec(GAT_SOURCE)
REL_TXT = torch.from_numpy(encode([f"{a} / {b}" for a, b in zip(rels.label_vi, rels.label_en)]))
LOCK = json.loads((OUT09 / "09_locked.json").read_text(encoding="utf-8"))
CFG = {a: {k: v for k, v in c.items() if k != "config"} for a, c in LOCK["cfg"].items()}
PRIMARY = LOCK.get("primary")
_SEL = FINAL / "results" / "DB15K" / "GAT_vs_AdaMF" / "method_selection.json"
if PRIMARY is None and _SEL.exists():           # 07c chạy sau 09: đọc thẳng kết quả chọn của 07c, không cần chạy lại 09
    PRIMARY = json.loads(_SEL.read_text(encoding="utf-8"))["primary"]
ALPHA = {tuple(k.split("|")): float(v) for k, v in LOCK["alpha"].items()}
ARCH_NAME = {"adamf": "AdaMF", "gat": "GAT"}
_xa = os.environ.get("MMKG_XAI_ARCHS", "primary")
XARCHS = ([PRIMARY] if PRIMARY else ["adamf", "gat"]) if _xa == "primary" else _xa.split(",")

def rebase(p):
    """Đường dẫn trong 09_locked.json là tuyệt đối; mở Drive ở máy khác thì ghép lại theo phần sau 'results/MMKG/'."""
    p = Path(p)
    if p.exists():
        return p
    s = str(p); k = s.find("results/MMKG/")
    q = OUT09 / s[k + len("results/MMKG/"):] if k >= 0 else p
    assert q.exists(), f"Không thấy {p}"
    return q

RUNS = {k: dict(v, dir=rebase(v["dir"])) for k, v in LOCK["runs"].items() if v["variant"] == "full"}
TEST_DIR = rebase(LOCK["test_dir"])

def build_model(cfg, arch, graph):
    kw = dict(ent_tot=N, rel_tot=R, dim=cfg["dim"], margin=cfg["margin"], epsilon=cfg["epsilon"], img_emb=train_visual, text_emb=textual)
    if arch == "adamf":
        return TrainOnlyAdaMF(availability=availability, **kw).to(DEVICE)
    return KGEdgeGAT(availability=availability, graph=graph, rel_text=REL_TXT, **kw).to(DEVICE)

def load_run(name):
    v = RUNS[name]; m = build_model(CFG[v["arch"]], v["arch"], TRAIN)
    st = torch.load(Path(v["dir"]) / "best_model.pt", map_location=DEVICE, weights_only=False)
    m.load_state_dict(st["model"]); m.eval().requires_grad_(False)
    return m, json.loads((Path(v["dir"]) / "complete.json").read_text())

@torch.no_grad()
def all_joint(model):
    model.eval()
    return model.joint(torch.arange(N, device=DEVICE))

def zscore(x):
    return (x - x.mean()) / max(float(x.std(ddof=1)), 1e-6) if len(x) > 1 else x * 0

def rank_of_row(score, i):
    return 1 + int((score > score[i]).sum())

def softmax_p(score, i):
    e = np.exp(score - score.max())
    return float(e[i] / e.sum())

class QCtx:
    """Một truy vấn gián tiếp: ứng viên = pool của split trừ ảnh bị lọc (như 09), điểm CLIP của câu, mặt nạ đáp án."""
    def __init__(self, q, sp, pos, drop, zq):
        self.q, self.sp, self.qid = q, sp, q.query_id
        rows = POOL[sp]; d = drop.get(q.query_id, set()) if drop else set()
        self.rows = rows[~np.isin(rows, list(d))] if d else rows
        self.ents = IMG_ENT[self.rows]; self.uents = np.unique(self.ents)
        self.clip = IMG_N[self.rows] @ zq; self.cz = zscore(self.clip)
        self.pos = np.isin(self.rows, pos.get(q.query_id, [])) if pos else np.zeros(len(self.rows), bool)
        self.anchor, self.rel, self.dir = int(q.anchor_entity_index), int(q.relation_index), q.prediction_direction
    def fused(self, g, alpha):
        return self.clip if alpha == 1.0 else alpha * self.cz + (1 - alpha) * zscore(g)

class RunScorer:
    """Một checkpoint 09 + cách chấm điểm đồ thị; với AdaMF có thêm hàm trộn phương thức để học lại nhúng cấu trúc của một thực thể."""
    def __init__(self, name):
        v = RUNS[name]; self.name, self.arch, self.seed = name, v["arch"], v["seed"]
        self.model, self.info = load_run(name); self.epoch = self.info["best_epoch"]; m = self.model
        self.alpha = ALPHA[(name, LANG)]; self.cfg = CFG[self.arch]
        self.margin, self.range = float(m.margin), float(m.ent_embedding_range)
        self.phase = float(m.rel_embedding_range.item()) / m.pi_const
        with torch.no_grad():
            self.J = all_joint(m); self.Rel = m.rel_embeddings.weight.detach().clone()
            if self.arch == "adamf":
                ids = torch.arange(N, device=DEVICE)
                self.Vp, self.Tp = m.img_proj(m.img_embeddings(ids)), m.text_proj(m.text_embeddings(ids))
                self.flags = torch.cat([torch.ones_like(m.availability[:, :1]), m.availability], 1)
    def dist(self, h, t, r):
        """Khoảng cách RotatE, broadcast được (bằng _calc của mô hình ở mode 'normal')."""
        ph = r / self.phase; rr, ri = torch.cos(ph), torch.sin(ph)
        hr, hi = torch.chunk(h, 2, -1); tr_, ti = torch.chunk(t, 2, -1)
        return torch.stack([hr * rr - hi * ri - tr_, hr * ri + hi * rr - ti], 0).norm(dim=0).sum(-1)
    def ent_score(self, ctx, Je, J=None):
        """Điểm (anchor, r, ·) hoặc (·, r, anchor) cho một hay nhiều nhúng Je."""
        a = (self.J if J is None else J)[ctx.anchor]; r = self.Rel[ctx.rel]
        h, t = (a, Je) if ctx.dir == "tail" else (Je, a)
        return self.margin - self.dist(h, t, r)
    @torch.no_grad()
    def graph(self, ctx, J=None, Rel=None):
        J = self.J if J is None else J; Rel = self.Rel if Rel is None else Rel
        e = torch.as_tensor(ctx.uents, device=DEVICE); a = J[ctx.anchor].expand(len(e), -1); r = Rel[ctx.rel].expand(len(e), -1)
        h, t = (a, J[e]) if ctx.dir == "tail" else (J[e], a)
        s = dict(zip(ctx.uents.tolist(), (self.margin - self.dist(h, t, r)).cpu().numpy()))
        return np.array([s[int(x)] for x in ctx.ents])
    # ----- AdaMF: trộn 3 phương thức (y hệt TrainOnlyAdaMF.joint) cho nhúng cấu trúc X của thực thể e -----
    def fuse(self, X, e):
        fl = self.flags[e]
        vals = torch.stack([X, self.Vp[e].expand_as(X), self.Tp[e].expand_as(X)], 1).masked_fill(~fl[None, :, None], 0.)
        lg = self.model.ent_attn(torch.tanh(vals)).squeeze(-1).masked_fill(~fl[None], float("-inf"))
        return (lg.softmax(-1).unsqueeze(-1) * vals).sum(1)
    def rot(self, x, r, sign=1.0):
        ph = sign * r / self.phase; c, s_ = torch.cos(ph), torch.sin(ph); xr, xi = torch.chunk(x, 2, -1)
        return torch.cat([xr * c - xi * s_, xr * s_ + xi * c], -1)
    def mimic(self, e, fids, keeps, overrides, seed):
        """E3 cấp 2 (Kelpie): học lại nhúng cấu trúc của e từ các bộ ba fids, mỗi biến thể giữ một tập con (keeps) và có thể thay đầu kia
        (overrides: {cột: thực thể mới}). Mọi biến thể cùng khởi tạo và cùng mẫu âm (cùng seed) nên chênh lệch chỉ do tập bộ ba.
        Phép quay RotatE giữ độ dài nên ‖e∘r − t‖ = ‖e − t∘r̄‖ và ‖h∘r − e‖ = ‖e − h∘r‖: tính sẵn đích c cho từng bộ ba / mẫu âm,
        mỗi bước chỉ còn ‖e − c‖ (cùng hàm mục tiêu với 10a, nhanh hơn nhiều)."""
        tr = torch.as_tensor(TRAIN[fids], device=DEVICE); is_head = tr[:, 0] == e
        partner = torch.where(is_head, tr[:, 1], tr[:, 0]); Rr = self.Rel[tr[:, 2]]
        sign = torch.where(is_head, -1.0, 1.0).to(Rr.dtype)[:, None]                       # e là đầu → nhân r̄; e là đuôi → nhân r
        g = torch.Generator().manual_seed(seed)
        neg = torch.randint(0, N, (len(fids), MIMIC["n_neg"]), generator=g).to(DEVICE)
        x0 = ((torch.rand(self.J.shape[1], generator=g) * 2 - 1) * self.range).to(DEVICE)
        Cn = self.rot(self.J[neg], (sign * Rr)[:, None, :])                                 # [F, n, d]
        kp_all = torch.as_tensor(np.stack(keeps), device=DEVICE)
        part = partner.repeat(len(kp_all), 1)
        for v, ov in enumerate(overrides):
            for c, x in (ov or {}).items():
                part[v, c] = int(x)
        Cp = self.rot(self.J[part], (sign * Rr)[None])                                      # [V, F, d]
        def d_(x, c):
            xr, xi = torch.chunk(x, 2, -1); cr, ci = torch.chunk(c, 2, -1)
            return torch.sqrt((xr - cr) ** 2 + (xi - ci) ** 2).sum(-1)
        temp, out = self.cfg["adv_temperature"], []
        step = max(1, int(MIMIC["max_elems"] // max(1, Cn.numel())))                       # số biến thể mỗi lô (giới hạn bộ nhớ)
        for a in range(0, len(kp_all), step):
            kw = kp_all[a:a + step].float(); den = kw.sum(1).clamp_min(1.); C = Cp[a:a + step]
            X = x0.repeat(len(kw), 1).clone().requires_grad_(True); opt = torch.optim.Adam([X], lr=MIMIC["lr"])
            with torch.enable_grad():
                for _ in range(MIMIC["steps"]):
                    Jx = self.fuse(X, e)
                    sp = self.margin - d_(Jx[:, None, :], C)                                     # [V, F]
                    sn = self.margin - d_(Jx[:, None, None, :], Cn[None])                        # [V, F, n]
                    ln = -(torch.softmax(sn * temp, -1).detach() * F.logsigmoid(-sn)).sum(-1)
                    loss = (((-F.logsigmoid(sp) + ln) / 2 * kw).sum(1) / den).sum()
                    opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                out.append(self.fuse(X, e).detach())
        return torch.cat(out)
    # ----- GAT: nhúng khi đồ thị truyền tin là `graph` (bỏ / thay cạnh), tham số giữ nguyên -----
    @torch.no_grad()
    def J_batch(self, graphs):
        """Như J_with cho nhiều đồ thị một lúc: ghép thành một đồ thị rời (mỗi biến thể một bản sao nút) → một lượt GATConv.
        Chép đúng KGEdgeGAT.encode; B1 của 10b kiểm tra trùng khớp với J_with."""
        m = self.model; V = len(graphs)
        if not hasattr(self, "_h0"):
            ids = torch.arange(N, device=DEVICE)
            vals = torch.stack([m.ent_embeddings(ids), m.img_proj(m.img_embeddings(ids)), m.text_proj(m.text_embeddings(ids))], 1)
            flags = torch.cat([torch.ones_like(m.availability[:, :1]), m.availability], 1).float()
            self._h0 = (vals * flags.unsqueeze(-1)).sum(1) / flags.sum(1, keepdim=True)
            self._ea = m.edge_projection(m.type_text); self._hin = m.in_proj(self._h0)
        src, dst, et = [], [], []
        for v, gr in enumerate(graphs):
            gr = torch.as_tensor(np.asarray(gr), dtype=torch.long, device=DEVICE).reshape(-1, 3); o = v * N
            src += [gr[:, 0] + o, gr[:, 1] + o]; dst += [gr[:, 1] + o, gr[:, 0] + o]; et += [gr[:, 2], gr[:, 2] + m.rel_tot]
        ei = torch.stack([torch.cat(src), torch.cat(dst)]); ea = self._ea[torch.cat(et)]
        h = self._hin.repeat(V, 1)
        h = m.norm1(h + F.elu(m.gat1(h, ei, edge_attr=ea))); h = m.norm2(h + F.elu(m.gat2(h, ei, edge_attr=ea)))
        return (self._h0.repeat(V, 1) + m.out_proj(h)).view(V, N, -1)
    def J_many(self, graphs, chunk=None):
        chunk = chunk or GAT_CHUNK
        for a in range(0, len(graphs), chunk):
            yield from self.J_batch(graphs[a:a + chunk])
    @torch.no_grad()
    def J_with(self, graph):
        m = self.model; g = torch.as_tensor(np.asarray(graph), dtype=torch.long, device=DEVICE).reshape(-1, 3)
        old = (m.edge_index, m.edge_type)
        m.edge_index = torch.stack([torch.cat([g[:, 0], g[:, 1]]), torch.cat([g[:, 1], g[:, 0]])])
        m.edge_type = torch.cat([g[:, 2], g[:, 2] + m.rel_tot])
        try:
            return m.encode()
        finally:
            m.edge_index, m.edge_type = old

# ---------- giải thích ----------
MIMIC = dict(steps=int(os.environ.get("XAI_STEPS", "300")), lr=0.01, n_neg=16, max_facts=256,   # như 10a
             max_elems=int(os.environ.get("XAI_MAX_ELEMS", "4000000")))
M_CAND = 24                      # AdaMF: số bộ ba ứng viên đo riêng (như 10a)
GAT_MAX_F = int(os.environ.get("XAI_GAT_MAXF", "256"))   # GAT: trần số cạnh trong vùng tiếp nhận 2 bước được che từng cái
K_GRID = [1, 2, 3, 5, 8]; COMP_TARGET = 0.80; N_RANDOM = 10
GAT_CHUNK = int(os.environ.get("XAI_GAT_CHUNK", "32"))   # số biến thể đồ thị mỗi lượt GAT

def hub_score(fid, e):
    h, t, _ = (int(x) for x in TRAIN[fid])
    return DEG[t] if h == e else DEG[h] if t == e else max(DEG[h], DEG[t])

def path_of(fid, e, a):
    """Đường đi ≤ 2 bước anchor … e* chứa bộ ba fid (danh sách fid theo thứ tự anchor → e*), hoặc None."""
    h, t, _ = (int(x) for x in TRAIN[fid]); ends = {h, t}
    if ends == {e, a}:
        return [fid]
    if e in ends and len(ends) == 2:
        x = (ends - {e}).pop()
        if x in NEIGH.get(a, ()):
            return [PAIR[(min(a, x), max(a, x))][0], fid]
    if a in ends and len(ends) == 2:
        x = (ends - {a}).pop()
        if x in NEIGH.get(e, ()):
            return [fid, PAIR[(min(x, e), max(x, e))][0]]
    return None

def perturb_triple(fid, keep, rng):
    """Thay đầu không giữ của bộ ba bằng thực thể từng đứng cùng vai trò với quan hệ đó; bộ ba mới không là bộ ba train/val/test."""
    h, t, r = (int(x) for x in TRAIN[fid]); repl_tail = h == keep
    pool = REL_TAILS[r] if repl_tail else REL_HEADS[r]; old = t if repl_tail else h
    for _ in range(50):
        x = int(rng.choice(pool)) if len(pool) > 2 else int(rng.integers(0, N))
        trip = (h, x, r) if repl_tail else (x, t, r)
        if x not in (keep, old) and trip not in TRAIN_KEYS and trip not in HELD_KEYS:
            return trip
    x = int(rng.integers(0, N))
    return (h, x, r) if repl_tail else (x, t, r)

def universe(rs, ctx, e, rng):
    """F: các bộ ba train có thể ảnh hưởng điểm của e*; cand: các bộ ba được đo riêng. n_all: |F| trước khi cắt trần."""
    a = ctx.anchor
    if rs.arch == "adamf":                                     # E3 cấp 2: bộ ba của chính e*
        fids = np.array(FACTS_OF.get(e, []), np.int64); n_all = len(fids)
        if not n_all:
            return fids, fids, 0
        tr = TRAIN[fids]; partner = np.where(tr[:, 0] == e, tr[:, 1], tr[:, 0])
        with torch.no_grad():
            T_ = torch.as_tensor(tr, device=DEVICE)
            fit = (rs.margin - rs.dist(rs.J[T_[:, 0]], rs.J[T_[:, 1]], rs.Rel[T_[:, 2]])).cpu().numpy()
        near = np.array([p in NEIGH.get(a, ()) for p in partner])
        prio = (partner == a) * 4 + near * 2 + (tr[:, 2] == ctx.rel) * 1
        cand = fids[np.lexsort((-fit, -prio))[:M_CAND]]
        if n_all > MIMIC["max_facts"]:
            rest = rng.choice(np.setdiff1d(fids, cand), MIMIC["max_facts"] - len(cand), replace=False)
            fids = np.concatenate([cand, rest])
        return fids, cand, n_all
    # GAT 2 lớp: điểm của e* phụ thuộc J[e*] và J[anchor]; J[v] phụ thuộc mọi cạnh chạm {v} ∪ N(v)
    nodes = {e, a} | NEIGH.get(e, set()) | NEIGH.get(a, set())
    fids = np.array(sorted({f for v in nodes for f in FACTS_OF.get(v, [])}), np.int64); n_all = len(fids)
    if n_all > GAT_MAX_F:
        tr = TRAIN[fids]
        prio = (tr[:, :2] == e).any(1) * 4 + (tr[:, :2] == a).any(1) * 2 + (tr[:, 2] == ctx.rel) * 1 + rng.random(n_all) * 0.5
        fids = np.sort(fids[np.argsort(-prio, kind="stable")[:GAT_MAX_F]])
    return fids, fids, n_all

def keep_node(fid, e, a):
    h, t, _ = (int(x) for x in TRAIN[fid])
    for v in (e, a):
        if v in (h, t):
            return v
    near = NEIGH.get(e, set()) | NEIGH.get(a, set())
    return h if h in near or t not in near else t

def variant_graphs(rs, ctx, e, fids, specs, seed, g0):
    """specs: [(bỏ: tập fid ⊂ F, thay: {fid: bộ ba mới})] → vector điểm đồ thị trên ứng viên cho từng biến thể.
    AdaMF: học lại nhúng của e* rồi áp chênh lệch giữa hai bản học lại vào điểm gốc (như 10a), các thực thể khác giữ nguyên.
    GAT: tính lại toàn bộ nhúng trên đồ thị đã sửa (mọi thực thể có thể đổi điểm)."""
    if rs.arch == "adamf":
        col = {int(f): i for i, f in enumerate(fids)}
        keeps, ovs = [np.ones(len(fids), bool)], [None]
        for rem, rep in specs:
            m = np.ones(len(fids), bool); m[[col[f] for f in rem]] = False; keeps.append(m)
            ovs.append({col[f]: (tr[1] if tr[0] == e else tr[0]) for f, tr in (rep or {}).items()})
        s = rs.ent_score(ctx, rs.mimic(e, fids, keeps, ovs, seed)).cpu().numpy()
        s_orig = float(rs.ent_score(ctx, rs.J[e])); out = []
        for sv in s[1:]:
            g = g0.copy(); g[ctx.ents == e] = s_orig - (float(s[0]) - float(sv)); out.append(g)
        return out
    ids, graphs = np.arange(len(TRAIN)), []
    for rem, rep in specs:
        T_ = TRAIN.copy()
        for f, tr in (rep or {}).items():
            T_[f] = tr
        graphs.append(T_[~np.isin(ids, list(rem))])
    return [rs.graph(ctx, J) for J in rs.J_many(graphs)]

def importance(rs, ctx, e, fids, cand, seed):
    """δ_f = điểm của e* giảm bao nhiêu khi bỏ riêng f (f ∈ cand); d_all = khi bỏ cả cand. Trả về (δ, d_all)."""
    if rs.arch == "adamf":
        col = {int(f): i for i, f in enumerate(fids)}; keeps = [np.ones(len(fids), bool)]
        for f in cand:
            m = np.ones(len(fids), bool); m[col[int(f)]] = False; keeps.append(m)
        m = np.ones(len(fids), bool); m[[col[int(f)] for f in cand]] = False; keeps.append(m)
        s = rs.ent_score(ctx, rs.mimic(e, fids, keeps, [None] * len(keeps), seed)).cpu().numpy()
        return float(s[0]) - s[1:-1], float(s[0]) - float(s[-1])
    s0 = float(rs.ent_score(ctx, rs.J[e])); ids = np.arange(len(TRAIN))
    s = [float(rs.ent_score(ctx, J[e], J)) for J in rs.J_many([TRAIN[ids != f] for f in cand] + [TRAIN[~np.isin(ids, cand)]])]
    return s0 - np.array(s[:-1]), s0 - s[-1]

def measure(ctx, rs, g, i):
    f = ctx.fused(g, rs.alpha); z = zscore(g)
    return dict(p=softmax_p(f, i), rank=rank_of_row(f, i), pg=softmax_p(z, i), rankg=rank_of_row(z, i), gi=float(g[i]))

def nrm(x, p0):
    return (p0 - x) / p0 if p0 > 0 else np.nan

def explain_query(rs, ctx, Ks, k_full, n_random=N_RANDOM, rand_curve=3, only_importance=False):
    """Giải thích ảnh hạng 1 (điểm cuối α·z(CLIP) + (1−α)·z(đồ thị)) rồi đo fidelity. Ks: các K cần đo; đủ biến thể chỉ ở K = k_full."""
    g0 = rs.graph(ctx); f0 = ctx.fused(g0, rs.alpha); i = int(np.argmax(f0)); e = int(ctx.ents[i]); a = ctx.anchor
    seed = stable_seed(ctx.qid, e, rs.name, rs.epoch); rng = np.random.default_rng(seed)
    fids, cand, n_all = universe(rs, ctx, e, rng)
    meta = dict(run=rs.name, arch=rs.arch, seed=rs.seed, query_id=ctx.qid, split=ctx.sp, direction=ctx.dir, relation=RLAB[ctx.rel],
                anchor=a, anchor_label=LAB[a], row=int(ctx.rows[i]), image_id=ROW2IMG[int(ctx.rows[i])], entity=e, entity_label=LAB[e],
                returned_is_relevant=bool(ctx.pos[i]), alpha=rs.alpha, n_facts_e=len(FACTS_OF.get(e, [])), n_universe=n_all,
                n_F=len(fids), n_cand=len(cand), rank_best_pos=(1 + int((f0 > f0[ctx.pos].max()).sum())) if ctx.pos.any() else np.nan)
    if not len(cand):
        return meta, [], None
    delta, d_all = importance(rs, ctx, e, fids, cand, seed); order = np.argsort(-delta, kind="stable")
    meta.update(d_all=d_all, n_positive_delta=int((delta > 0).sum()))
    S_of = {K: [int(cand[j]) for j in order[:K] if delta[j] > 0] for K in Ks}
    S_of = {K: S for K, S in S_of.items() if S}
    top = [dict(fid=int(cand[j]), delta=float(delta[j]), text=fact_text(int(cand[j]))) for j in order[:max(K_GRID)]]
    expl = dict(meta, K=k_full, S=S_of.get(k_full, []), universe=[int(f) for f in fids], top=top)
    if only_importance or not S_of:                              # chọn K* (10a, C6): độ giảm điểm của e* khi bỏ đồng thời top-K / khi bỏ cả cand
        gv = variant_graphs(rs, ctx, e, fids, [(set(S), None) for S in S_of.values()], seed, g0) if S_of else []
        return meta, [dict(K=K, size_S=len(S), dscore=float(g0[i] - g[i]), comp=float(g0[i] - g[i]) / d_all if d_all > 0 else np.nan)
                      for (K, S), g in zip(S_of.items(), gv)], expl
    specs, tags = [], []
    def add(rem, rep, tag):
        specs.append((set(rem), rep)); tags.append(tag)
    Fset = [int(f) for f in fids]; hubs = sorted(Fset, key=lambda f: (-hub_score(f, e), f))
    for K, S in S_of.items():
        full, k = K == k_full, len(S)
        add(S, None, ("minus_top", K, 0)); add(set(Fset) - set(S), None, ("only_top", K, 0))
        if full:
            add([], {f: perturb_triple(f, keep_node(f, e, a), rng) for f in S}, ("perturb_top", K, 0))
            add(hubs[:k], None, ("minus_deg", K, 0))
        for rr in range(n_random if full else rand_curve):
            rc = [int(x) for x in rng.choice(Fset, k, replace=False)]
            add(rc, None, ("minus_rand", K, rr))
            if full:
                add(set(Fset) - set(rc), None, ("only_rand", K, rr))
    base = measure(ctx, rs, g0, i)
    res = {t: measure(ctx, rs, g, i) for t, g in zip(tags, variant_graphs(rs, ctx, e, fids, specs, seed, g0))}
    NA = dict(p=np.nan, rank=np.nan, pg=np.nan, rankg=np.nan, gi=np.nan); p0, pg0, r0, rg0 = base["p"], base["pg"], base["rank"], base["rankg"]
    rows = []
    for K, S in S_of.items():
        g = lambda kind: res.get((kind, K, 0), NA)
        mr = [v for t, v in res.items() if t[0] == "minus_rand" and t[1] == K]; orr = [v for t, v in res.items() if t[0] == "only_rand" and t[1] == K]
        fp, fm = nrm(g("minus_top")["p"], p0), nrm(g("only_top")["p"], p0)
        c1, c2 = min(max(fp, 0.), 1.), min(max(fm, 0.), 1.)
        paths = [path_of(f, e, a) for f in S]
        rows.append(dict(
            K=K, size_S=len(S), sparsity_F=1 - len(S) / max(1, n_all), dscore_minus_top=base["gi"] - g("minus_top")["gi"],
            comp=(base["gi"] - g("minus_top")["gi"]) / d_all if d_all > 0 else np.nan, dscore_rand=float(np.mean([base["gi"] - x["gi"] for x in mr])),
            p0=p0, rank0=r0, pg0=pg0, rankg0=rg0,
            fid_plus=fp, fid_minus=fm, characterization=(2 * c1 * (1 - c2) / (c1 + 1 - c2)) if (c1 + 1 - c2) > 0 else np.nan,
            drank_minus_top=g("minus_top")["rank"] - r0, PN=float(g("minus_top")["rank"] > 1), PS=float(g("only_top")["rank"] == 1),
            fid_perturb=nrm(g("perturb_top")["p"], p0), drank_perturb=g("perturb_top")["rank"] - r0,
            PN_perturb=float(g("perturb_top")["rank"] > 1) if K == k_full else np.nan,
            fid_plus_deg=nrm(g("minus_deg")["p"], p0), drank_deg=g("minus_deg")["rank"] - r0,
            PN_deg=float(g("minus_deg")["rank"] > 1) if K == k_full else np.nan,
            fid_plus_rand=float(np.mean([nrm(x["p"], p0) for x in mr])), drank_rand=float(np.mean([x["rank"] - r0 for x in mr])),
            PN_rand=float(np.mean([x["rank"] > 1 for x in mr])),
            fid_minus_rand=float(np.mean([nrm(x["p"], p0) for x in orr])) if orr else np.nan,
            PS_rand=float(np.mean([x["rank"] == 1 for x in orr])) if orr else np.nan,
            g_fid_plus=nrm(g("minus_top")["pg"], pg0), g_fid_minus=nrm(g("only_top")["pg"], pg0),
            g_fid_plus_rand=float(np.mean([nrm(x["pg"], pg0) for x in mr])), g_drank_minus_top=g("minus_top")["rankg"] - rg0,
            on_path=float(np.mean([p is not None for p in paths])), touches_e=float(np.mean([e in TRAIN[f, :2] for f in S])),
            share_derived=float(np.mean([int(TRAIN[f, 2]) in DERIVED for f in S])),
            heldout_overlap=int(sum(tuple(int(v) for v in TRAIN[f]) in HELD_KEYS for f in S)),
            in_train=int(all(tuple(int(v) for v in TRAIN[f]) in TRAIN_KEYS for f in S)),
            S_fids=" ".join(map(str, S)), S_text=" | ".join(fact_text(f) for f in S)))
    return meta, rows, expl

def explanation_text(expl):
    e, a = expl["entity"], expl["anchor"]; lines = []
    for f in expl["S"]:
        d = next(x["delta"] for x in expl["top"] if x["fid"] == f); p = path_of(f, e, a)
        lines.append(f"{fact_text(f)}  (δ = {d:+.3f}" + (f"; đường đi: {' ⇒ '.join(fact_text(x) for x in p)})" if p and len(p) > 1 else
                                                         "; nối trực tiếp anchor và e*)" if p else "; ngoài đường đi anchor–e*)"))
    return lines

def thumb_b64(row, size=180):
    from PIL import Image
    im = Image.open(image_path(row)).convert("RGB"); im.thumbnail((size, size))
    b = io.BytesIO(); im.save(b, "JPEG", quality=80)
    return base64.b64encode(b.getvalue()).decode()
SHARED_HASH = "0730184495b314a7"
print(f"N={N} R={R} train={len(TRAIN)} | phương pháp chính: {PRIMARY} | giải thích: {XARCHS} | thiết bị {DEVICE}")

## Giao diện dùng chung (nhúng nguyên văn `src/multimodal_xai/demo_web.py`)

In [ ]:
# Giao diện web demo dùng chung cho ba cấp độ (Visual Genome, DB15K / FB15K-237-IMG, ViHeritage-MMKG).
#
# Mỗi notebook demo chỉ tạo dữ liệu (header + danh sách kết quả); trang HTML, đồ thị lực kiểu Obsidian (force-graph),
# màu sắc và cách đọc giống nhau ở cả ba cấp. Màu theo báo cáo (report/figures/scripts/make_figures.py):
#   xanh ngọc = thực thể neo, cam = ảnh / thực thể của ảnh trả về, xanh dương = phần tử của tập giải thích S,
#   xanh nhạt = đường nối về neo (chỉ để đọc, không đo), xám nét đứt = quan hệ được hỏi, xám = nút khác.
# Màn hình chính chỉ hiện phần cần cho demo; số liệu phụ nằm trong mục "Chi tiết" (mặc định đóng).
# File này được nhúng nguyên văn vào notebook demo cấp 1 và cấp 3; notebook cấp 2 import từ package.
#
# Dữ liệu một kết quả (item):
#   rank, title, correct (True / False / None), img (data URL), graph = {nodes, links} (dùng node(), link()),
#   empty (chữ hiện khi không có đồ thị), why = [(nhãn, nội dung)] (kiểm chứng chính, tối đa 3 dòng),
#   more = [(nhãn, nội dung)] (chi tiết), bars = [{label, w, text}] (chi tiết), note (chi tiết),
#   row = {clip_rank, removed} (bảng xếp hạng),
#   story = một câu trên đồ thị trả lời "Vì sao trả về X?" (bằng chứng mạnh nhất, đường đi về neo),
#   removal = câu hiện khi bấm "Bỏ giải thích S" (hạng trước, sau; so với bỏ ngẫu nhiên).
import base64
import html as _html
import io
import json
import os
import urllib.request
from pathlib import Path

PALETTE = {"anchor": "#1baf7a", "cand": "#eb6834", "other": "#d9d8d4",
           "fact": "#2a78d6", "path": "#9cc0ef", "query": "#8f8e8a"}
PAGE_HEIGHT = 720

_FG_URLS = ["https://cdn.jsdelivr.net/npm/force-graph@1.52.0/dist/force-graph.min.js",
            "https://unpkg.com/force-graph@1.52.0/dist/force-graph.min.js"]
_FG = {}


def force_graph_js():
    """Nhúng thẳng force-graph vào trang (iframe srcdoc không tải được script ngoài ở một số trình duyệt)."""
    if "js" not in _FG:
        _FG["js"] = None
        local = os.environ.get("XAI_FORCEGRAPH_JS")
        if local and Path(local).exists():
            _FG["js"] = Path(local).read_text(encoding="utf-8")
        else:
            for u in _FG_URLS:
                try:
                    _FG["js"] = urllib.request.urlopen(u, timeout=30).read().decode("utf-8")
                    break
                except Exception:
                    continue
    return f"<script>{_FG['js']}</script>" if _FG["js"] else f'<script src="{_FG_URLS[0]}"></script>'


def image_data_url(img, size=640, quality=85):
    """PIL.Image hoặc đường dẫn → data URL JPEG; None nếu không mở được."""
    try:
        from PIL import Image
        if img is None:
            return None
        if not hasattr(img, "convert"):
            img = Image.open(img)
        img = img.convert("RGB")
        img.thumbnail((size, size))
        b = io.BytesIO()
        img.save(b, format="JPEG", quality=quality)
        return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode()
    except Exception:
        return None


def rank_text(r):
    if r is None:
        return "không có"
    r = float(r)
    return f"{int(r)}" if r == int(r) else f"{r:.1f}"


def _move(r0, x):
    if x is None:
        return "không đo"
    if abs(float(x) - r0) < 0.05:
        return f"vẫn hạng {r0}"
    return f"{'xuống' if x > r0 else 'lên'} hạng {rank_text(x)}"


def removal_sentence(n, unit, r0, after, rand=None):
    """Câu hiện khi bấm "Bỏ giải thích S": hạng sau khi bỏ S, so với bỏ cùng số phần tử ngẫu nhiên (trung bình)."""
    s = f"Bỏ {n} {unit} giải thích: ảnh {_move(r0, after)}"
    if rand is not None:
        s += f" · bỏ {n} {unit} ngẫu nhiên: {_move(r0, rand)}" + (" (trung bình)" if float(rand) != int(rand) else "")
    return s


def node(nid, name, kind, val=2.0, tip=None, result=False):
    """result=True: nút này cũng là thực thể của ảnh trả về (ví dụ mô hình trả về chính thực thể neo), vẽ bằng bức ảnh."""
    return dict(id=nid, name=str(name), kind=kind, val=float(val), tip=str(tip or name), result=bool(result))


def link(source, target, rel, kind, delta=0.0, label=None, tip=None):
    return dict(source=source, target=target, rel=str(rel), kind=kind, delta=float(delta),
                label=str(rel if label is None else label), tip=str(tip or rel))


DEFAULT_LEGEND = {"anchor": "neo", "cand": "ảnh trả về", "other": "nút khác",
                  "fact": "giải thích S (dày theo Δ)", "path": "nối về neo", "query": "quan hệ được hỏi"}

_CSS = r"""
:root{--bg:#1e1e1e;--panel:#252525;--panel2:#2b2b2b;--line:#353535;--text:#e3e3e0;--muted:#9a9a96;--accent:#2a78d6;--good:#1baf7a;--bad:#eb6834}
body.light{--bg:#ffffff;--panel:#f6f6f4;--panel2:#ececea;--line:#dcdbd7;--text:#1f1f1d;--muted:#6b6a66}
*{box-sizing:border-box}html,body{height:100%}
body{margin:0;background:var(--bg);color:var(--text);font:14px/1.45 system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;display:flex;flex-direction:column;overflow:hidden}
.hdr{padding:10px 14px 8px;border-bottom:1px solid var(--line);display:flex;flex-wrap:wrap;align-items:baseline;gap:4px 18px}
.q{font-size:16px;font-weight:600}.meta{font-size:13.5px;color:var(--muted)}.meta b{color:var(--text);font-weight:600}
.tg{font-size:14px;font-weight:600;margin-left:auto}
.tabs{display:flex;gap:2px;padding:2px 10px 0;border-bottom:1px solid var(--line)}
.tab{background:none;border:0;color:var(--muted);padding:7px 14px;font:inherit;font-weight:600;cursor:pointer;border-bottom:2px solid transparent}
.tab.on{color:var(--text);border-bottom-color:var(--accent)}
.pane{display:none;flex:1;min-height:0;padding:10px 12px}.pane.on{display:block}
.gv{display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1.4fr);gap:12px;height:100%}
.viewer{display:flex;flex-direction:column;min-width:0;min-height:0;overflow:auto;padding-right:4px}
.vhead{margin-bottom:6px;font-size:15px}
.ok{color:var(--good);font-weight:700}.no{color:var(--bad);font-weight:700}
.stage{position:relative;background:var(--panel);border-radius:8px;display:flex;align-items:center;justify-content:center;height:290px;flex:none;overflow:hidden}
.stage img{max-width:100%;max-height:290px;display:block}.ph{color:var(--muted);font-size:13px}
.nav{position:absolute;top:50%;transform:translateY(-50%);background:var(--panel2);border:1px solid var(--line);color:var(--text);width:34px;height:34px;border-radius:50%;cursor:pointer;font-size:18px;line-height:30px;z-index:3;opacity:.9}
.nav:hover{border-color:var(--accent)}.nav.prev{left:8px}.nav.next{right:8px}
.thumbs{display:flex;gap:6px;margin-top:8px;flex:none}
.thumbs div{flex:1;height:48px;border-radius:5px;background:var(--panel2) center/cover no-repeat;cursor:pointer;border:2px solid transparent;opacity:.6;position:relative}
.thumbs div.on{border-color:var(--accent);opacity:1}
.thumbs span{position:absolute;left:3px;top:1px;font-size:11px;font-weight:700;color:#fff;text-shadow:0 0 3px #000}
.box{margin-top:10px;background:var(--panel);border:1px solid var(--line);border-radius:8px;padding:8px 12px;flex:none}
.box h4{margin:0 0 4px;font-size:12px;color:var(--muted);font-weight:600}
.why{font-size:14.5px;margin:3px 0}.why .l{color:var(--muted)}.why .v{font-weight:700}
details.more{margin-top:8px;font-size:12.5px;color:var(--muted);flex:none}details.more summary{cursor:pointer}
.kv{display:grid;grid-template-columns:auto 1fr;gap:2px 10px;margin-top:4px}.kv span:nth-child(even){color:var(--text)}
.bar{display:flex;align-items:center;gap:8px;margin:3px 0}.bar .n{width:96px}
.bar i{display:block;height:7px;background:var(--accent);border-radius:4px;flex:none}.bar span:last-child{color:var(--text)}
.gwrap{position:relative;border-radius:8px;overflow:hidden;background:var(--bg);border:1px solid var(--line);min-width:0}
#graph{position:absolute;inset:0}
.empty{position:absolute;inset:0;display:none;align-items:center;justify-content:center;text-align:center;color:var(--muted);padding:28px;font-size:14px}
.gear{position:absolute;top:8px;right:8px;z-index:4;background:var(--panel2);border:1px solid var(--line);color:var(--text);border-radius:6px;padding:4px 9px;cursor:pointer;font:inherit;font-size:13px}
.panel{position:absolute;top:42px;right:8px;width:260px;z-index:4;background:var(--panel);border:1px solid var(--line);border-radius:8px;padding:10px 12px;display:none;font-size:12.5px}
.panel.on{display:block}
.panel label{display:flex;justify-content:space-between;align-items:center;margin:5px 0;gap:8px}
.panel input[type=range]{width:100px;accent-color:var(--accent)}.panel input[type=checkbox]{accent-color:var(--accent)}
.panel button{width:100%;margin-top:8px;background:var(--panel2);border:1px solid var(--line);color:var(--text);border-radius:5px;padding:5px;cursor:pointer;font:inherit}
.story{position:absolute;left:8px;top:8px;right:150px;z-index:3;background:var(--panel);opacity:.96;border:1px solid var(--line);border-radius:8px;padding:6px 11px;pointer-events:none}
.story .t{font-size:15.5px;font-weight:700}.story .s{font-size:13.5px;margin-top:2px}
.rmS{position:absolute;top:8px;right:48px;z-index:4;background:var(--accent);border:0;color:#fff;border-radius:6px;padding:5px 11px;cursor:pointer;font:inherit;font-size:13px;font-weight:600}
.rmS.on{background:var(--panel2);color:var(--text);border:1px solid var(--line)}
.banner{position:absolute;left:50%;transform:translateX(-50%);top:78px;z-index:3;background:var(--panel);border:2px solid var(--bad);border-radius:8px;padding:7px 14px;font-size:14.5px;font-weight:600;display:none;text-align:center;max-width:90%}
.legend{position:absolute;left:8px;bottom:8px;z-index:3;font-size:12.5px;color:var(--text);background:var(--panel);opacity:.95;padding:4px 10px;border-radius:6px;border:1px solid var(--line);pointer-events:none;display:flex;flex-wrap:wrap;gap:2px 14px;max-width:calc(100% - 16px)}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin:0 5px -1px 0}
.ln{display:inline-block;width:20px;height:0;margin:0 5px 3px 0;border-top:3px solid}
table{width:100%;border-collapse:collapse;font-size:14px}th{text-align:left;color:var(--muted);font-weight:600;font-size:13px;padding:7px 10px;border-bottom:1px solid var(--line)}
td{padding:7px 10px;border-bottom:1px solid var(--line)}tbody tr{cursor:pointer}tbody tr:hover{background:var(--panel)}
td .th{width:84px;height:56px;border-radius:4px;background:var(--panel2) center/cover no-repeat}
.up{color:var(--good)}.down{color:var(--bad)}
#p-rank{overflow:auto}
.float-tooltip-kap{background:var(--panel)!important;color:var(--text)!important;border:1px solid var(--line);font-size:12.5px!important;padding:4px 8px!important;border-radius:5px}
"""

_BODY = r"""
<div class="hdr" id="hdr"></div>
<div class="tabs"><button class="tab on" data-t="graph">Đồ thị giải thích</button><button class="tab" data-t="rank">Bảng xếp hạng</button></div>
<div class="pane on" id="p-graph"><div class="gv">
 <div class="viewer">
  <div class="vhead" id="vh"></div>
  <div class="stage"><button class="nav prev" title="Ảnh trước (←)">‹</button><div id="stage-img"></div><button class="nav next" title="Ảnh sau (→)">›</button></div>
  <div class="thumbs" id="thumbs"></div>
  <div id="info"></div>
 </div>
 <div class="gwrap"><div id="graph"></div><div class="empty"></div><div class="story" id="story"></div>
  <button class="rmS" id="rmS"></button><div class="banner" id="banner"></div><button class="gear">⚙</button>
  <div class="panel">
   <label>Nền sáng<input type="checkbox" data-k="light"></label>
   <label>Nhãn quan hệ<input type="checkbox" data-k="labels" checked></label><label>Tên nút<input type="checkbox" data-k="names" checked></label>
   <label>Mũi tên<input type="checkbox" data-k="arrows" checked></label><label>Đường nối về neo<input type="checkbox" data-k="paths" checked></label>
   <label>Kích thước nút<input type="range" data-k="size" min="0.5" max="2" step="0.1" value="1"></label><label>Độ dày cạnh<input type="range" data-k="width" min="0.3" max="3" step="0.1" value="1"></label>
   <label>Lực đẩy<input type="range" data-k="repel" min="20" max="800" step="10" value="160"></label>
   <label>Độ dài cạnh<input type="range" data-k="dist" min="20" max="260" step="5" value="95"></label><button id="fit">Căn vừa khung</button></div>
  <div class="legend" id="legend"></div></div></div></div>
<div class="pane" id="p-rank"><table><thead><tr id="rank-head"></tr></thead><tbody id="rows"></tbody></table></div>
"""

_JS = r"""
const $=s=>document.querySelector(s),$$=s=>[...document.querySelectorAll(s)];
const esc=t=>String(t??'').replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const P=D.palette;
const THEME={dark:{bg:'#1e1e1e',text:'#e3e3e0',muted:'#9a9a96',label:'#9cc0ef',qlabel:'#b8b7b2'},
             light:{bg:'#ffffff',text:'#1f1f1d',muted:'#6b6a66',label:'#1c5fae',qlabel:'#52514e'}};
const S={light:false,labels:true,names:true,arrows:true,paths:true,size:1,width:1,repel:160,dist:95};
let cur=0,hover=null,nb={},fg=null;const gel=$('#graph');
const T=()=>THEME[S.light?'light':'dark'];
function rgba(hex,a){const n=parseInt(hex.slice(1),16);return `rgba(${n>>16&255},${n>>8&255},${n&255},${a})`;}
function tab(t){$$('.tab').forEach(b=>b.classList.toggle('on',b.dataset.t===t));$$('.pane').forEach(p=>p.classList.toggle('on',p.id==='p-'+t));
  if(t==='graph'&&fg)fg.width(gel.clientWidth).height(gel.clientHeight);}
$$('.tab').forEach(b=>b.onclick=()=>tab(b.dataset.t));
// ---------- header: câu truy vấn, neo / quan hệ, hạng ảnh đúng ----------
(function(){const h=D.header;let s='<span class="q">“'+esc(h.query)+'”</span>';
  if(h.chips&&h.chips.length)s+='<span class="meta">'+h.chips.map(c=>esc(c[0])+': <b>'+esc(c[1])+'</b>').join(' · ')+'</span>';
  if(h.target)s+='<span class="tg">'+esc(h.target)+'</span>';$('#hdr').innerHTML=s;})();
// ---------- đồ thị ----------
const rad=n=>Math.sqrt(n.val)*4*S.size;
function mid(l){const s=l.source,t=l.target,c=l.__controlPoints;
  if(c&&c.length===2)return{x:.25*s.x+.5*c[0]+.25*t.x,y:.25*s.y+.5*c[1]+.25*t.y};
  if(c&&c.length===4)return{x:.125*s.x+.375*c[0]+.375*c[2]+.125*t.x,y:.125*s.y+.375*c[1]+.375*c[3]+.125*t.y};
  return{x:(s.x+t.x)/2,y:(s.y+t.y)/2};}
function active(l){return hover===null||l.source.id===hover||l.target.id===hover;}
const gone=l=>S.removed&&l.kind==='fact';
function lcol(l){if(gone(l))return rgba(P.query,.35);const c=P[l.kind]||P.fact,a=active(l);return rgba(c,l.kind==='path'?(a?.85:.15):(a?.95:.15));}
const IMG={};function imgOf(src){if(!src)return null;if(!IMG[src]){const im=new Image();im.src=src;IMG[src]=im;}return IMG[src];}
function collide(){let nodes=[];const f=alpha=>{for(let i=0;i<nodes.length;i++)for(let j=i+1;j<nodes.length;j++){
    const a=nodes[i],b=nodes[j];let dx=b.x-a.x,dy=b.y-a.y,d=Math.hypot(dx,dy)||1e-3;const m=rad(a)+rad(b)+34;
    if(d<m){const k=(m-d)/d*.5*Math.min(1,alpha*4);dx*=k;dy*=k;
      if(a.fx===undefined){a.x-=dx;a.y-=dy;}if(b.fx===undefined){b.x+=dx;b.y+=dy;}}}};f.initialize=n=>{nodes=n;};return f;}
function initGraph(){ if(fg||!window.ForceGraph)return;
  fg=ForceGraph()(gel).backgroundColor(T().bg).autoPauseRedraw(false).nodeId('id').nodeVal(n=>n.val).nodeRelSize(4)
   .nodeLabel(n=>esc(n.tip)).linkLabel(l=>esc(l.tip)).linkVisibility(l=>l.kind!=='path'||S.paths)
   .linkColor(lcol).linkCurvature(l=>l.curv||0)
   .linkWidth(l=>(gone(l)?1:l.kind==='fact'?1.4+4*Math.min(1,Math.abs(l.delta)/(D.items[cur].dmax||1)):(l.kind==='query'?1.4:1))*S.width)
   .linkLineDash(l=>gone(l)?[3,5]:l.kind==='query'?[5,4]:(l.kind==='path'?[2,3]:null))
   .linkDirectionalArrowLength(l=>S.arrows&&!gone(l)?(l.kind==='fact'?6:4.5):0).linkDirectionalArrowRelPos(1).linkDirectionalArrowColor(lcol)
   .linkCanvasObjectMode(()=>'after').linkCanvasObject((l,ctx,s)=>{
      if(!S.labels||s<0.35||!l.label||(l.kind==='path'&&!S.paths))return;
      const m=mid(l),f=Math.max(11.5/s,2.5);ctx.font=f+'px system-ui,sans-serif';ctx.textAlign='center';ctx.textBaseline='middle';
      const w=ctx.measureText(l.label).width;ctx.globalAlpha=gone(l)?.3:active(l)?1:.25;
      ctx.fillStyle=T().bg;ctx.fillRect(m.x-w/2-3/s,m.y-f/2-1.5/s,w+6/s,f+3/s);
      ctx.fillStyle=l.kind==='query'?T().qlabel:(l.kind==='path'?T().muted:T().label);ctx.fillText(l.label,m.x,m.y);ctx.globalAlpha=1;})
   .nodeCanvasObject((n,ctx,s)=>{const r=rad(n),act=hover===null||(nb[hover]&&nb[hover].has(n.id));
      ctx.globalAlpha=act?1:.25;ctx.beginPath();ctx.arc(n.x,n.y,r+2/s,0,2*Math.PI);ctx.fillStyle=T().bg;ctx.fill();
      ctx.beginPath();ctx.arc(n.x,n.y,r,0,2*Math.PI);ctx.fillStyle=P[n.kind]||P.other;ctx.fill();
      const im=n.img&&imgOf(n.img);
      if(im&&im.complete&&im.naturalWidth){ctx.save();ctx.beginPath();ctx.arc(n.x,n.y,r,0,2*Math.PI);ctx.clip();
        const k=Math.max(2*r/im.naturalWidth,2*r/im.naturalHeight);ctx.drawImage(im,n.x-im.naturalWidth*k/2,n.y-im.naturalHeight*k/2,im.naturalWidth*k,im.naturalHeight*k);
        ctx.restore();ctx.beginPath();ctx.arc(n.x,n.y,r,0,2*Math.PI);ctx.lineWidth=4/s;ctx.strokeStyle=P.cand;ctx.stroke();}
      if(n.id===hover){ctx.lineWidth=2.5/s;ctx.strokeStyle=T().text;ctx.stroke();}
      ctx.globalAlpha=1;if(!S.names&&n.id!==hover)return;
      const f=Math.max(12.5/s,3);ctx.font=(n.kind!=='other'?'600 ':'')+f+'px system-ui,sans-serif';ctx.textAlign='center';ctx.textBaseline='top';
      ctx.lineWidth=3.5/s;ctx.strokeStyle=T().bg;ctx.lineJoin='round';ctx.strokeText(n.name,n.x,n.y+r+3/s);
      ctx.fillStyle=act?T().text:rgba(T().text,.3);ctx.fillText(n.name,n.x,n.y+r+3/s);
      const rk=D.roles.cand+' '+D.items[cur].rank;
      const role=n.role||(n.kind==='anchor'?(n.result?D.roles.anchor+', cũng là '+rk:D.roles.anchor):n.kind==='cand'?rk:'');
      if(role){const f2=Math.max(11/s,2.5);ctx.font=f2+'px system-ui,sans-serif';ctx.strokeText(role,n.x,n.y+r+5/s+f);
        ctx.fillStyle=n.kind==='anchor'?P.anchor:P.cand;ctx.fillText(role,n.x,n.y+r+5/s+f);}})
   .nodePointerAreaPaint((n,c,ctx)=>{ctx.fillStyle=c;ctx.beginPath();ctx.arc(n.x,n.y,rad(n)+4,0,2*Math.PI);ctx.fill();})
   .onNodeHover(n=>{hover=n?n.id:null;gel.style.cursor=n?'pointer':null;})
   .cooldownTicks(160).onEngineStop(()=>fg.zoomToFit(400,90));
  forces();}
function forces(){if(!fg)return;fg.d3Force('charge').strength(-S.repel);fg.d3Force('collide',collide());
  fg.d3Force('link').distance(l=>l.kind==='fact'?S.dist:S.dist*1.25);fg.nodeRelSize(4*S.size);fg.d3ReheatSimulation();}
function seedLayout(g,fix){// bố cục ban đầu cố định (nan quạt quanh nút neo hoặc nút ảnh) để mỗi lần mở đều ra cùng một hình
  const others=g.nodes.filter(n=>n!==fix),cands=others.filter(n=>n.kind==='cand'),rest=others.filter(n=>n.kind!=='cand'),R=S.dist*1.7;
  cands.forEach((n,i)=>{n.x=(i-(cands.length-1)/2)*S.dist;n.y=S.dist*1.2;});
  rest.forEach((n,i)=>{const a=cands.length?(-Math.PI*0.1-Math.PI*0.8*(i+.5)/rest.length):(2*Math.PI*i/rest.length-Math.PI/2);
    n.x=R*Math.cos(a)*(cands.length?1.25:1);n.y=R*Math.sin(a);});}
function curvatures(g){const grp={};
  g.links.forEach(l=>{const a=String(l.source),b=String(l.target),k=a<b?a+'|'+b:b+'|'+a;(grp[k]=grp[k]||[]).push(l);});
  Object.values(grp).forEach(arr=>arr.forEach((l,i)=>{
    if(String(l.source)===String(l.target)){l.curv=1+.4*i;return;}
    if(arr.length===1){l.curv=0;return;}
    let c=(i-(arr.length-1)/2)*.8;if(String(l.source)>String(l.target))c=-c;l.curv=c;}));}
function legend(it){const L=D.legend,kinds=new Set(it.graph.nodes.map(n=>n.kind)),lk=new Set(it.graph.links.map(l=>l.kind));const r=[];
  ['anchor','cand','other'].forEach(k=>{if(kinds.has(k)&&L[k])r.push('<span><span class="dot" style="background:'+P[k]+'"></span>'+esc(L[k])+'</span>');});
  if(lk.has('fact')&&L.fact)r.push('<span><span class="ln" style="border-color:'+P.fact+'"></span>'+esc(L.fact)+'</span>');
  if(lk.has('path')&&L.path)r.push('<span><span class="ln" style="border-color:'+P.path+';border-top-style:dotted"></span>'+esc(L.path)+'</span>');
  if(lk.has('query')&&L.query)r.push('<span><span class="ln" style="border-color:'+P.query+';border-top-style:dashed"></span>'+esc(L.query)+'</span>');
  $('#legend').innerHTML=r.join('');$('#legend').style.display=r.length?'flex':'none';}
// ---------- cột trái: chỉ phần cần cho demo, số liệu phụ trong "Chi tiết" ----------
function badge(it){return it.correct===true?' <span class="ok">✓ ảnh đúng</span>':(it.correct===false?' <span class="no">✗ chưa đúng</span>':'');}
function info(it){let s='';
  if(it.why&&it.why.length)s+='<div class="box"><h4>'+esc(D.check_title)+'</h4>'+it.why.map(x=>'<div class="why"><span class="l">'+esc(x[0])+':</span> <span class="v">'+esc(x[1])+'</span></div>').join('')+'</div>';
  let m='';if(it.more&&it.more.length)m+='<div class="kv">'+it.more.map(x=>'<span>'+esc(x[0])+'</span><span>'+esc(x[1])+'</span>').join('')+'</div>';
  if(it.bars&&it.bars.length)m+='<div style="margin-top:6px">'+esc(D.bar_title)+'</div>'+it.bars.map(b=>'<div class="bar"><span class="n">'+esc(b.label)+'</span><i style="width:'+Math.max(3,Math.round(90*b.w))+'px"></i><span>'+esc(b.text)+'</span></div>').join('');
  if(it.note)m+='<div style="margin-top:6px">'+esc(it.note)+'</div>';
  if(m)s+='<details class="more"><summary>Chi tiết</summary>'+m+'</details>';return s;}
function show(i){cur=(i+D.items.length)%D.items.length;const it=D.items[cur];hover=null;
  $('#vh').innerHTML='<b>#'+it.rank+' · '+esc(it.title)+'</b>'+badge(it);
  $('#stage-img').innerHTML=it.img?'<img src="'+it.img+'" alt="'+esc(it.title)+'">':'<span class="ph">không mở được ảnh</span>';
  $('#info').innerHTML=info(it);$$('.thumbs div').forEach((t,k)=>t.classList.toggle('on',k===cur));
  const has=it.graph.links.length>0;$('.empty').style.display=has?'none':'flex';$('.empty').textContent=it.empty||'Không có phần tử đồ thị nào để vẽ cho ảnh này.';
  $('#story').innerHTML=has?'<div class="t">'+esc(D.story_title.replace('{x}',it.title))+'</div>'+(it.story?'<div class="s">'+esc(it.story)+'</div>':''):'';
  $('#story').style.display=has?'block':'none';
  S.removed=false;$('#banner').style.display='none';const rb=$('#rmS');rb.classList.remove('on');rb.textContent=D.remove_label;
  rb.style.display=has&&it.removal?'block':'none';
  legend(it);initGraph();if(!fg)return;
  const g=JSON.parse(JSON.stringify(it.graph));nb={};const fix=g.nodes.find(n=>n.kind==='anchor')||g.nodes.find(n=>n.kind==='cand');
  g.nodes.forEach(n=>{nb[n.id]=new Set([n.id]);if((n.kind==='cand'||n.result)&&it.img){n.img=it.img;n.val=30;}if(fix&&n.id===fix.id){n.fx=0;n.fy=0;n.x=0;n.y=0;}});seedLayout(g,fix);
  g.links.forEach(l=>{if(nb[l.source])nb[l.source].add(l.target);if(nb[l.target])nb[l.target].add(l.source);});curvatures(g);
  fg.width(gel.clientWidth).height(gel.clientHeight).graphData(has?g:{nodes:[],links:[]});
  const c0=cur;[60,700,1800].forEach(t=>setTimeout(()=>{if(fg&&cur===c0){fg.width(gel.clientWidth).height(gel.clientHeight);fg.zoomToFit(0,90);}},t));}
$('#thumbs').innerHTML=D.items.map((it,k)=>'<div title="'+esc(it.title)+'" style="background-image:url('+(it.img||'')+')" onclick="show('+k+')"><span>#'+it.rank+(it.correct?' ✓':'')+'</span></div>').join('');
$('.prev').onclick=()=>show(cur-1);$('.next').onclick=()=>show(cur+1);
document.addEventListener('keydown',e=>{if(!$('#p-graph').classList.contains('on'))return;if(e.key==='ArrowLeft')show(cur-1);if(e.key==='ArrowRight')show(cur+1);});
$('.gear').onclick=()=>$('.panel').classList.toggle('on');
$('#rmS').onclick=()=>{const it=D.items[cur];S.removed=!S.removed;const rb=$('#rmS');rb.classList.toggle('on',S.removed);
  rb.textContent=S.removed?D.restore_label:D.remove_label;$('#banner').textContent=it.removal||'';$('#banner').style.display=S.removed?'block':'none';};
$$('.panel input').forEach(inp=>inp.oninput=()=>{const k=inp.dataset.k;S[k]=inp.type==='checkbox'?inp.checked:+inp.value;
  if(k==='light'){document.body.classList.toggle('light',S.light);if(fg)fg.backgroundColor(T().bg);}
  if(['repel','dist','size'].includes(k))forces();});
$('#fit').onclick=()=>fg&&fg.zoomToFit(400,90);
// ---------- bảng xếp hạng ----------
$('#rank-head').innerHTML=['Hạng','Ảnh','Kết quả','Ảnh đúng','Hạng nếu chỉ dùng CLIP',D.removed_col].map(x=>'<th>'+esc(x)+'</th>').join('');
$('#rows').innerHTML=D.items.map((it,k)=>{const r=it.row||{},cr=r.clip_rank,d=(cr??it.rank)-it.rank;
  return '<tr onclick="tab(\'graph\');show('+k+')"><td><b>#'+it.rank+'</b></td><td><div class="th" style="background-image:url('+(it.img||'')+')"></div></td><td>'+esc(it.title)+'</td><td>'+(it.correct===true?'<span class="ok">✓</span>':'')+'</td><td>'+(cr==null?'':'#'+esc(cr))+(d>0?' <span class="up">▲'+d+'</span>':d<0?' <span class="down">▼'+(-d)+'</span>':'')+'</td><td>'+esc(r.removed??'')+'</td></tr>';}).join('');
window.addEventListener('resize',()=>fg&&fg.width(gel.clientWidth).height(gel.clientHeight));
(function(){const h=String(location.hash||'');if(h.includes('light')){S.light=true;document.body.classList.add('light');$('.panel input[data-k=light]').checked=true;}
  const m=h.match(/i=(\d+)/);if(m)cur=+m[1];if(h.includes('rank'))tab('rank');if(h.includes('more'))setTimeout(()=>$$('details.more').forEach(d=>d.open=true),0);
  if(h.includes('rm'))setTimeout(()=>$('#rmS').click(),50);})();
show(cur);if(fg)fg.backgroundColor(T().bg);
"""


def _prepare(items):
    for it in items:
        g = it.setdefault("graph", {"nodes": [], "links": []})
        it["dmax"] = max([abs(l["delta"]) for l in g["links"] if l["kind"] == "fact"] + [1e-12])
    return items


def results_page(header, items, legend=None, check_title="Kiểm chứng: bỏ giải thích rồi chấm lại",
                 bar_title="Che từng phương thức rồi chấm lại:", removed_col="Bỏ S → hạng", title="Demo truy vấn có giải thích",
                 story_title="Vì sao trả về {x}?", remove_label="Bỏ giải thích S", restore_label="Khôi phục S",
                 roles=None):
    """header: dict(query, chips=[(nhãn, giá trị)], target). items: danh sách kết quả theo hạng (xem đầu file).
    Mỗi item có thêm: story (một câu nêu bằng chứng mạnh nhất), removal (câu hiện khi bấm "Bỏ giải thích S")."""
    data = dict(header=header, items=_prepare(items), palette=PALETTE, legend={**DEFAULT_LEGEND, **(legend or {})},
                check_title=check_title, bar_title=bar_title, removed_col=removed_col, story_title=story_title,
                remove_label=remove_label, restore_label=restore_label,
                roles={"anchor": "neo của truy vấn", "cand": "ảnh hạng", **(roles or {})})
    payload = json.dumps(data, ensure_ascii=False).replace("</", "<\\/")
    return (f"<!DOCTYPE html><html lang='vi'><head><meta charset='utf-8'><title>{_html.escape(title)}</title><style>{_CSS}</style>"
            f"{force_graph_js()}</head><body>{_BODY}<script>const D={payload};{_JS}</script></body></html>")


def results_iframe(header, items, height=PAGE_HEIGHT, **kw):
    page = results_page(header, items, **kw)
    return (f'<iframe srcdoc="{_html.escape(page, quote=True)}" style="width:100%;height:{height}px;border:0;'
            f'border-radius:8px;background:#1e1e1e"></iframe>')


def message_html(text):
    return f'<div style="padding:14px;color:#9a9a96;font:14px system-ui,sans-serif">{_html.escape(text)}</div>'


def launch(demo, in_colab, height=PAGE_HEIGHT + 160):
    """Colab: mặc định tạo link công khai (*.gradio.live, tự hết hạn) để mở trang ở tab riêng khi quay video.
    Đặt XAI_DEMO_SHARE=0 để chỉ hiện trong notebook. Tắt notebook sau khi quay để link ngừng hoạt động."""
    share = os.environ.get("XAI_DEMO_SHARE", "1" if in_colab else "0") == "1"
    demo.queue()
    if share:
        return demo.launch(share=True, inline=False, show_error=True)
    return demo.launch(share=False, inline=in_colab, height=height, show_error=True)

# Gom các hàm của giao diện vào W (giống `from multimodal_xai import demo_web as W` ở notebook cấp 2)
import types as _types
W = _types.SimpleNamespace(PALETTE=PALETTE, PAGE_HEIGHT=PAGE_HEIGHT, image_data_url=image_data_url, rank_text=rank_text, removal_sentence=removal_sentence, node=node, link=link, results_page=results_page, results_iframe=results_iframe, message_html=message_html, launch=launch)

## Web demo

In [ ]:
# Web demo cấp 3 (ViHeritage-MMKG, câu tiếng Việt). Nhúng nguyên văn vào demo_cap3_viheritage.ipynb, chạy sau ô X0 (mã dùng chung
# của 10b, 10c, 10d: dữ liệu 08b, checkpoint 09, hai bộ giải thích) và demo_web.py.
# Giải thích ảnh nào thì đo đúng ảnh đó: S = tối đa K* bộ ba train có δ dương lớn nhất (AdaMF: học lại nhúng kiểu Kelpie;
# GAT: che cạnh, chính xác), rồi bỏ S, làm nhiễu S, bỏ cùng số bộ ba ngẫu nhiên (10 lần) và xếp hạng lại như 10b.
import os
from types import SimpleNamespace

ARCHS = [a for a in ["gat", "adamf"] if f"{a}_full_s{MAIN_SEED}" in RUNS]
SCORER = {a: RunScorer(f"{a}_full_s{MAIN_SEED}") for a in ARCHS}
_info = XOUT / "10b_run_info.json"
K_OF = json.loads(_info.read_text(encoding="utf-8")).get("K_star", {}) if _info.exists() else {}
K_OF = {a: int(K_OF.get(a, {"adamf": 1, "gat": 8}[a])) for a in ARCHS}
MODEL_LABEL = {a: f"CLIP + {ARCH_NAME[a]}" for a in ARCHS}
VARIANTS = {"Câu hỏi": "text_vi_1", "Câu mô tả": "text_vi_2", "Tiếng Anh": "text_en"}
QT, POS, DROP = split_queries("test")
QROW = QT.set_index("query_id")
print("Mô hình:", {a: f"α_vi = {ALPHA[(SCORER[a].name, 'vi')]}, K* = {K_OF[a]}" for a in ARCHS})

def alpha_of(arch, variant):
    rs = SCORER[arch]; lang = "en" if variant == "text_en" else "vi"
    return float(ALPHA.get((rs.name, lang), rs.alpha))

def test_query(qid):
    return SimpleNamespace(query_id=qid, **QROW.loc[qid].to_dict())

# ---------- giải thích ảnh ở vị trí i (cùng thứ tự gọi bộ sinh ngẫu nhiên với explain_query của 10b) ----------
def explain_at(rs, ctx, i, K, alpha):
    g0 = rs.graph(ctx); e, a = int(ctx.ents[i]), ctx.anchor
    seed = stable_seed(ctx.qid, e, rs.name, rs.epoch); rng = np.random.default_rng(seed)
    fids, cand, n_all = universe(rs, ctx, e, rng)
    if not len(cand):
        return dict(e=e, S=[])
    delta, d_all = importance(rs, ctx, e, fids, cand, seed); order = np.argsort(-delta, kind="stable")
    S = [int(cand[j]) for j in order[:K] if delta[j] > 0]
    dS = {int(cand[j]): float(delta[j]) for j in order[:K]}
    if not S:
        return dict(e=e, S=[])
    Fset, k = [int(f) for f in fids], len(S)
    specs = [(set(S), None), (set(), {f: perturb_triple(f, keep_node(f, e, a), rng) for f in S})]
    specs += [(set(int(x) for x in rng.choice(Fset, k, replace=False)), None) for _ in range(N_RANDOM)]
    gs = variant_graphs(rs, ctx, e, fids, specs, seed, g0)
    rank = lambda g: rank_of_row(ctx.fused(g, alpha), i)
    return dict(e=e, S=S, delta=dS, r0=rank(g0), r_rm=rank(gs[0]), r_pert=rank(gs[1]), r_rand=float(np.mean([rank(g) for g in gs[2:]])),
                dscore=float(g0[i] - gs[0][i]), n_all=n_all)

def _sent(f):
    h, t, r = (int(x) for x in TRAIN[f]); return f"{LAB[h]} {RLAB[r]} {LAB[t]}"

def _arrow(f):
    h, t, r = (int(x) for x in TRAIN[f]); return f"{LAB[h]} -[{RLAB[r]}]→ {LAB[t]}"

def vh_graph(ctx, ex):
    e, a, S = ex["e"], ctx.anchor, ex["S"]
    nodes = {e: W.node(e, LAB[e], "cand", 7, f"{LAB[e]} (thực thể của ảnh)")}
    nodes[a] = W.node(a, LAB[a], "anchor", 9, f"{LAB[a]} (neo của truy vấn)", result=(a == e))
    src, dst = (a, e) if ctx.dir == "tail" else (e, a); rl = RLAB[ctx.rel]
    links = [W.link(src, dst, rl, "query", 0, f"{rl} ?", f"Quan hệ được hỏi: {LAB[src]} -[{rl}]→ {LAB[dst]}")]
    vmax = max(ex["delta"][f] for f in S); size, seen = {}, set(S)
    for f in S:
        h, t, r = (int(x) for x in TRAIN[f]); d = ex["delta"][f]
        for n in (h, t):
            nodes.setdefault(n, W.node(n, LAB[n], "other", 1.5)); size[n] = size.get(n, 0) + d
        links.append(W.link(h, t, RLAB[r], "fact", d, f"{RLAB[r]} ({d:.3g})", f"{_arrow(f)} · Δ = {d:+.4f}"))
    for f in S:
        for f2 in (path_of(f, e, a) or []):
            if f2 in seen:
                continue
            seen.add(f2); h, t, r = (int(x) for x in TRAIN[f2])
            for n in (h, t):
                nodes.setdefault(n, W.node(n, LAB[n], "other", 1.5))
            links.append(W.link(h, t, RLAB[r], "path", 0, RLAB[r], f"{_arrow(f2)} · nối về neo, không đo"))
    for n, v in size.items():
        if nodes[n]["kind"] == "other":
            nodes[n]["val"] = 1.5 + 6 * v / vmax
    f = S[0]; p = path_of(f, e, a)
    if a == e:
        story = f"Ảnh trả về chính là thực thể neo. Bằng chứng mạnh nhất: {_sent(f)}."
    elif p and len(p) > 1:
        story = f"Bằng chứng mạnh nhất: {_sent(f)}, và {_sent([x for x in p if x != f][0])} (nối về neo)."
    else:
        story = f"Bằng chứng mạnh nhất: {_sent(f)}" + (" (nối thẳng với neo)." if p else " (không nối về neo).")
    return dict(nodes=list(nodes.values()), links=links), story

# ---------- xếp hạng một truy vấn ----------
def make_ctx(q, z, gt_qid=None):
    pos = {q.query_id: POS.get(gt_qid, [])} if gt_qid else None
    drop = {q.query_id: DROP.get(gt_qid, set())} if gt_qid else None
    return QCtx(q, "test", pos, drop, z)

def rank_query(arch, q, text, gt_qid=None, k=5, variant="text_vi_1"):
    rs, K, alpha = SCORER[arch], K_OF[arch], alpha_of(arch, variant)
    ctx = make_ctx(q, encode([text])[0], gt_qid); g0 = rs.graph(ctx); f0 = ctx.fused(g0, alpha)
    order = np.argsort(-f0, kind="stable")[:int(k)]; has_gt = bool(ctx.pos.any())
    items = []
    for rank, i in enumerate(order.tolist(), 1):
        e = int(ctx.ents[i]); row = int(ctx.rows[i])
        it = dict(rank=rank, title=LAB[e], correct=bool(ctx.pos[i]) if has_gt else None, img=W.image_data_url(image_path(row)),
                  row=dict(clip_rank=rank_of_row(ctx.clip, i), removed=""),
                  more=[("Hạng nếu chỉ dùng CLIP", str(rank_of_row(ctx.clip, i))), ("Điểm đồ thị", f"{float(g0[i]):+.3f}"),
                        ("Điểm cuối", f"{float(f0[i]):+.3f}"),
                        ("Cách tính", f"{alpha:.1f}·z(CLIP) + {1 - alpha:.1f}·z(đồ thị), K* = {K}"),
                        ("Bộ giải thích", "học lại nhúng của thực thể (AdaMF)" if arch == "adamf" else "che cạnh trên đồ thị truyền tin (GAT)")])
        ex = explain_at(rs, ctx, i, K, alpha)
        if not ex["S"]:
            it["graph"] = dict(nodes=[], links=[])
            it["empty"] = "Không bộ ba train nào làm tăng điểm của thực thể này, nên không có giải thích dạng bộ ba."
        else:
            n = len(ex["S"]); r0 = ex["r0"]
            it["graph"], it["story"] = vh_graph(ctx, ex)
            it["why"] = [(f"Bỏ {n} bộ ba giải thích S", f"hạng {r0} → {W.rank_text(ex['r_rm'])}"),
                         (f"Bỏ {n} bộ ba ngẫu nhiên", f"hạng {r0} → {W.rank_text(ex['r_rand'])}")]
            it["removal"] = W.removal_sentence(n, "bộ ba", r0, ex["r_rm"], ex["r_rand"])
            it["row"]["removed"] = W.rank_text(ex["r_rm"])
            it["more"] += [("Làm nhiễu S (thay đầu kia)", f"hạng {r0} → {ex['r_pert']}"), ("Bỏ S: điểm đồ thị giảm", f"{ex['dscore']:.3f}")]
        items.append(it)
    chips = [("Neo", LAB[ctx.anchor]), ("Quan hệ", RLAB[ctx.rel] + (" (tìm đầu)" if ctx.dir == "head" else " (tìm đuôi)"))]
    target = None
    if has_gt:
        c = 1 + int((ctx.clip > ctx.clip[ctx.pos].max()).sum()); m = 1 + int((f0 > f0[ctx.pos].max()).sum())
        target = f"Ảnh đúng: CLIP hạng {c} → {MODEL_LABEL[arch]} hạng {m}"
    return dict(query=text, chips=chips, target=target), items

def rank_direct(q, text, k=5):
    """Truy vấn trực tiếp: theo thiết kế chỉ xếp hạng bằng CLIP, không có giải thích đồ thị."""
    rows = POOL["test"]; d = DROP.get(q.query_id, set())
    rows = rows[~np.isin(rows, list(d))] if d else rows
    clip = IMG_N[rows] @ encode([text])[0]; order = np.argsort(-clip, kind="stable")[:int(k)]; P = np.isin(rows, POS.get(q.query_id, []))
    items = [dict(rank=r, title=LAB[int(IMG_ENT[rows[i]])], correct=bool(P[i]) if P.any() else None, img=W.image_data_url(image_path(int(rows[i]))),
                  graph=dict(nodes=[], links=[]), row=dict(clip_rank=r, removed=""),
                  empty="Truy vấn trực tiếp: theo thiết kế chỉ xếp hạng bằng CLIP, không bộ ba nào tham gia, nên không có giải thích đồ thị.")
             for r, i in enumerate(order.tolist(), 1)]
    target = f"Ảnh đúng: CLIP hạng {1 + int((clip > clip[P].max()).sum())}" if P.any() else None
    return dict(query=text, chips=[("Loại", "trực tiếp")], target=target), items

def page(header, items):
    return W.results_iframe(header, items)

# ---------- 5 + 5 ca của báo cáo (D4) ----------
def _case_label(arch, qid):
    q = test_query(qid); text = qtext(q)
    if q.query_type == "direct":
        return f"{text}"
    rs, alpha = SCORER[arch], alpha_of(arch, "text_vi_1")
    ctx = make_ctx(q, encode([text])[0], qid); f0 = ctx.fused(rs.graph(ctx), alpha)
    if not ctx.pos.any():
        return text
    c = 1 + int((ctx.clip > ctx.clip[ctx.pos].max()).sum()); m = 1 + int((f0 > f0[ctx.pos].max()).sum())
    return f"{text}  (CLIP {c} → {m})"

DEMO_CASES = {a: {} for a in ARCHS}
_cases = XOUT / "D4_cases.csv"
if _cases.exists():
    _df = pd.read_csv(_cases)
    for a in ARCHS:
        sub = _df[(_df.arch == a) & _df.query_id.isin(QROW.index)]
        sub = pd.concat([sub[sub.loai == "thành công"], sub[sub.loai != "thành công"]])
        for r in sub.drop_duplicates("query_id").itertuples():
            DEMO_CASES[a][f"[{r.loai}] {_case_label(a, r.query_id)}"] = r.query_id
for a in ARCHS:                                            # ca Huế của hình trong báo cáo lên đầu
    hue = [c for c, q in DEMO_CASES[a].items() if "Huế có những di tích" in c]
    if hue:
        DEMO_CASES[a] = {**{hue[0]: DEMO_CASES[a][hue[0]]}, **DEMO_CASES[a]}
print({a: len(v) for a, v in DEMO_CASES.items()}, "ca trong báo cáo")

_RESULT = {}
def run_case(model, variant_name, choice, k=5):
    arch = next(a for a in ARCHS if MODEL_LABEL[a] == model); variant = VARIANTS[variant_name]
    qid = DEMO_CASES[arch].get(choice)
    if qid is None:
        return W.message_html("Chọn một ca trong danh sách.")
    key = ("case", arch, variant, qid, int(k))
    if key not in _RESULT:
        q = test_query(qid); text = qtext(q, variant)
        _RESULT[key] = page(*(rank_direct(q, text, k) if q.query_type == "direct" else rank_query(arch, q, text, qid, k, variant)))
    return _RESULT[key]

_n = lambda x: unicodedata.normalize("NFC", str(x)).lower().strip()
ENT_CHOICES = sorted({LAB[i] for i in range(N)}); REL_CHOICES = [RLAB[r] for r in sorted(RLAB)]
NAME2E = {_n(LAB[i]): i for i in range(N)}; NAME2R = {_n(RLAB[r]): r for r in RLAB}
def run_free(model, text, anchor, relation, direction, k=5):
    arch = next(a for a in ARCHS if MODEL_LABEL[a] == model)
    text = (text or "").strip(); a = NAME2E.get(_n(anchor)); r = NAME2R.get(_n(relation))
    if not text or a is None or r is None:
        return W.message_html("Nhập câu truy vấn, chọn thực thể neo và quan hệ.")
    d = "head" if direction.startswith("head") else "tail"
    hit = QT[(QT.query_type == "indirect") & (QT.anchor_entity_index == a) & (QT.relation_index == r) & (QT.prediction_direction == d)]
    gt = hit.query_id.iloc[0] if len(hit) else None              # trùng một truy vấn test: biết ảnh đúng
    key = ("free", arch, text, a, r, d, int(k))
    if key not in _RESULT:
        q = SimpleNamespace(query_id=f"demo|{text}|{a}|{r}|{d}", anchor_entity_index=a, relation_index=r, prediction_direction=d)
        variant = "text_en" if text.isascii() else "text_vi_1"
        _RESULT[key] = page(*rank_query(arch, q, text, gt, k, variant))
    return _RESULT[key]

# ---------- giao diện ----------
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True); import gradio as gr
_m0 = MODEL_LABEL[ARCHS[0]]; _cs0 = list(DEMO_CASES[ARCHS[0]])
with gr.Blocks(title="Truy vấn đa phương thức có giải thích bằng đồ thị") as demo:
    gr.Markdown("### Cấp độ 3 · ViHeritage-MMKG (tiếng Việt, nhóm tự xây)")
    with gr.Row(equal_height=True):
        model_in = gr.Radio([MODEL_LABEL[a] for a in ARCHS], value=_m0, label="Mô hình", scale=1)
        var_in = gr.Radio(list(VARIANTS), value="Câu hỏi", label="Cách diễn đạt câu", scale=2)
        k_in = gr.Slider(3, 10, value=5, step=1, label="Top-k", scale=1)
    case_in = gr.Dropdown(_cs0, value=_cs0[0] if _cs0 else None, label="Ca trong báo cáo (5 thành công, 5 thất bại)")
    with gr.Accordion("Tự nhập truy vấn", open=False):
        with gr.Row(equal_height=True):
            text_in = gr.Textbox(label="Câu truy vấn", placeholder="Ảnh một ngôi chùa cổ ở Hà Nội", scale=4)
            anchor_in = gr.Dropdown(ENT_CHOICES, value="Huế" if "Huế" in ENT_CHOICES else ENT_CHOICES[0], label="Thực thể neo", filterable=True, scale=2)
            rel_in = gr.Dropdown(REL_CHOICES, value=REL_CHOICES[0], label="Quan hệ", filterable=True, scale=2)
            dir_in = gr.Radio(["head: (?, quan hệ, neo)", "tail: (neo, quan hệ, ?)"], value="head: (?, quan hệ, neo)", label="Chiều", scale=2)
            free_btn = gr.Button("Tìm", variant="primary", scale=1)
    out = gr.HTML()
    def _on_model(model):
        arch = next(a for a in ARCHS if MODEL_LABEL[a] == model); cs = list(DEMO_CASES[arch])
        return gr.update(choices=cs, value=cs[0] if cs else None)
    model_in.change(_on_model, model_in, case_in)
    case_args = [model_in, var_in, case_in, k_in]
    case_in.change(run_case, case_args, out); var_in.change(run_case, case_args, out); k_in.release(run_case, case_args, out)
    free_btn.click(run_free, [model_in, text_in, anchor_in, rel_in, dir_in, k_in], out)
    demo.load(run_case, case_args, out)

if os.environ.get("XAI_DEMO_LAUNCH", "1") == "1":
    W.launch(demo, IN_COLAB)